# تبيّن · Tabayyan — interactive demo

Saudi-aware PII detection & redaction. All values below are **synthetic**.

```bash
pip install tabayyan        # core, zero runtime deps
pip install 'tabayyan[presidio]'  # optional Presidio integration
```

## 1. Detect

In [ ]:
from tabayyan import scan

text = ('اسم المريض عبدالله القحطاني، الهوية ١١٥٨٨١٣٩٩٦، '
        'جوال +966512345678، الآيبان SA9886987973091141707536, MRN: A0099231')

for m in scan(text):
    print(f'{m.entity_type.value:22} {m.confidence.value:6} {m.category.value:18} {m.value}')

## 2. Redact — mask / partial / tokenize (reversible)

In [ ]:
from tabayyan import scan_and_redact, restore, RedactionMode

print(scan_and_redact(text, RedactionMode.MASK).text)
print(scan_and_redact(text, RedactionMode.PARTIAL, partial_keep_last=4).text)

r = scan_and_redact(text, RedactionMode.TOKENIZE)
print(r.text)
print('restored == original:', restore(r.text, r.vault) == text)

## 3. Middleware + audit (cross-border flagging)

Flags a transfer when personal data heads to an endpoint outside the Kingdom.

In [ ]:
from tabayyan import Guard, AuditLog

guard = Guard(in_kingdom_hosts=['llm.myhospital.health.sa'], audit=AuditLog())
pr = guard.protect(text, destination='https://contoso.openai.azure.com')
print('redacted :', pr.text)
print('cross_border_transfer:', pr.audit.cross_border_transfer)
print('health_data_present  :', pr.audit.health_data_present)
print('audit:', pr.audit.to_json())

## 4. Inside Presidio (optional)

Requires `pip install 'tabayyan[presidio]'`.

In [ ]:
# from presidio_analyzer import AnalyzerEngine
# from tabayyan.integrations.presidio import register_saudi_recognizers
# analyzer = AnalyzerEngine()
# register_saudi_recognizers(analyzer)
# analyzer.analyze(text='national ID 1158813996', language='en')